In [ ]:
import pandas as pd
import pandas as pd
from shapely.geometry import LineString
from shapely.wkt import dumps
import numpy as np
from scipy.spatial import cKDTree


def network_update(
    base_node_df: pd.DataFrame,
    base_link_df: pd.DataFrame,
    merge_node_df: pd.DataFrame,
    merge_link_df: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Integrates and updates node and link IDs from a base network and a network to be merged.

    Args:
        base_node_df (pd.DataFrame): DataFrame of nodes for the base network.
        base_link_df (pd.DataFrame): DataFrame of links for the base network.
        merge_node_df (pd.DataFrame): DataFrame of nodes for the network to be merged.
        merge_link_df (pd.DataFrame): DataFrame of links for the network to be merged.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
            A tuple containing:
            - updated_base_node_df (pd.DataFrame): Updated base node DataFrame.
            - updated_merge_node_df (pd.DataFrame): Updated merged node DataFrame.
            - updated_base_link_df (pd.DataFrame): Updated base link DataFrame.
            - updated_merge_link_df (pd.DataFrame): Updated merged link DataFrame.
    """

    # 1. Integrate and Update Node IDs
    # Save existing node_id to 'old_node_id' column for both base and merge dataframes.
    # If 'old_node_id' already exists, it will be overwritten.
    base_node_df['old_node_id'] = base_node_df['node_id']
    merge_node_df['old_node_id'] = merge_node_df['node_id']

    # Base network's node_ids remain as they are (assuming they start from 1).
    # New node_ids for the merged network start after the maximum node_id of the base network.
    num_rows_in_base_df = len(base_node_df) # Get the total number of rows (nodes) in the base DataFrame.
    max_node_id_in_base_df = base_node_df['node_id'].max() # Get the maximum value in the 'node_id' column.

    # Compare the max_node_id with the number of rows.
    # If they are not equal, the node_ids are not sequential from 1.
    is_base_renumbering_needed = (max_node_id_in_base_df != num_rows_in_base_df)
    if is_base_renumbering_needed:
        print(f"base_node_df: Max ID ({max_node_id_in_base_df}) != Row count ({num_rows_in_base_df}). Renumbering required.")
        # Sort the DataFrame by the existing 'node_id' to maintain a logical order.
        sorted_base_df = base_node_df.sort_values(by='node_id')
        # Reset the index of the sorted DataFrame to ensure it's clean (0, 1, 2, ...).
        reset_base_df = sorted_base_df.reset_index(drop=True)
        # Create a new array of sequential numbers, from 1 to the number of rows.
        new_sequential_ids_for_base = np.arange(1, num_rows_in_base_df + 1)
        
        # Assign the new sequential IDs to the 'node_id' column.
        # We use .copy() to avoid potential pandas SettingWithCopyWarning.
        base_node_df = reset_base_df.copy()
        base_node_df['node_id'] = new_sequential_ids_for_base


    num_rows_in_merge_df = len(merge_node_df) # Get the total number of rows (nodes) in the merge DataFrame.
    max_node_id_in_merge_df = merge_node_df['node_id'].max() # Get the maximum value in the 'node_id' column.

    # Compare the max_node_id with the number of rows.
    is_merge_renumbering_needed = (max_node_id_in_merge_df != num_rows_in_merge_df)
    if is_merge_renumbering_needed:
        print(f"merge_node_df: Max ID ({max_node_id_in_merge_df}) != Row count ({num_rows_in_merge_df}). Renumbering required.")
        # Sort, reset index, and create new sequential IDs.
        sorted_merge_df = merge_node_df.sort_values(by='node_id')
        reset_merge_df = sorted_merge_df.reset_index(drop=True)
        new_sequential_ids_for_merge = np.arange(1, num_rows_in_merge_df + 1)
        
        # Assign the new IDs.
        merge_node_df = reset_merge_df.copy()
        merge_node_df['node_id'] = new_sequential_ids_for_merge

    max_base_node_id = base_node_df['node_id'].max()
    merge_node_df['node_id'] = merge_node_df['node_id'] + max_base_node_id

    # Create a node ID mapping dictionary (old_node_id -> new_node_id)
    # For the base network
    node_id_map = pd.Series(base_node_df['node_id'].values, index=base_node_df['old_node_id']).to_dict()
    # For the merge network
    merge_node_id_map = pd.Series(merge_node_df['node_id'].values, index=merge_node_df['old_node_id']).to_dict()
    

    # 2. Integrate and Update Link IDs
    # Save existing link_id to 'old_link_id' column for both base and merge dataframes.
    # If 'old_link_id' already exists, it will be overwritten.
    base_link_df['old_link_id'] = base_link_df['link_id']
    merge_link_df['old_link_id'] = merge_link_df['link_id']

    num_rows_in_base_link_df = len(base_link_df) # Get the total number of rows (links) in the base DataFrame.
    max_link_id_in_base_link_df = base_link_df['link_id'].max() # Get the maximum value in the 'link_id' column.

    # Compare the max_link_id with the number of rows.
    # If they are not equal, the link_ids are not sequential from 1.
    is_base_link_renumbering_needed = (max_link_id_in_base_link_df != num_rows_in_base_link_df)

    if is_base_link_renumbering_needed:
        print(f"base_link_df: Max ID ({max_link_id_in_base_link_df}) != Row count ({num_rows_in_base_link_df}). Renumbering required.")
        
        # Sort the DataFrame by the existing 'link_id' to maintain a logical order.
        sorted_base_link_df = base_link_df.sort_values(by='link_id')
        
        # Reset the index of the sorted DataFrame to ensure it's clean (0, 1, 2, ...).
        reset_base_link_df = sorted_base_link_df.reset_index(drop=True)
        
        # Create a new array of sequential numbers, from 1 to the number of rows.
        new_sequential_ids_for_base_link = np.arange(1, num_rows_in_base_link_df + 1)
        
        # Assign the new sequential IDs to the 'link_id' column.
        # We use .copy() to avoid potential pandas SettingWithCopyWarning.
        base_link_df = reset_base_link_df.copy()
        base_link_df['link_id'] = new_sequential_ids_for_base_link

    num_rows_in_merge_link_df = len(merge_link_df) # Get the total number of rows (links) in the merge DataFrame.
    max_link_id_in_merge_link_df = merge_link_df['link_id'].max() # Get the maximum value in the 'link_id' column.

    # Compare the max_link_id with the number of rows.
    is_merge_link_renumbering_needed = (max_link_id_in_merge_link_df != num_rows_in_merge_link_df)

    if is_merge_link_renumbering_needed:
        print(f"merge_link_df: Max ID ({max_link_id_in_merge_link_df}) != Row count ({num_rows_in_merge_link_df}). Renumbering required.")
        
        # Sort, reset index, and create new sequential IDs.
        sorted_merge_link_df = merge_link_df.sort_values(by='link_id')
        reset_merge_link_df = sorted_merge_link_df.reset_index(drop=True)
        new_sequential_ids_for_merge_link = np.arange(1, num_rows_in_merge_link_df + 1)
        
        # Assign the new IDs.
        merge_link_df = reset_merge_link_df.copy()
        merge_link_df['link_id'] = new_sequential_ids_for_merge_link

    # New link_ids for the merged network start after the maximum link_id of the base network.
    max_base_link_id = base_link_df['link_id'].max()
    merge_link_df['link_id'] = merge_link_df['link_id'] + max_base_link_id

    # 3. Update from_node_id and to_node_id in Link Files
    # Update base link file using the node ID map
    base_link_df['from_node_id'] = base_link_df['from_node_id'].map(node_id_map)
    base_link_df['to_node_id'] = base_link_df['to_node_id'].map(node_id_map)

    # Update merged link file using the node ID map
    merge_link_df['from_node_id'] = merge_link_df['from_node_id'].map(merge_node_id_map)
    merge_link_df['to_node_id'] = merge_link_df['to_node_id'].map(merge_node_id_map)

    return base_node_df, merge_node_df, base_link_df, merge_link_df



In [ ]:
#import os
#os.chdir("file path")

base_node_df = pd.read_csv('node_updated.csv')
base_link_df = pd.read_csv('link_updated.csv')
merge_node_df = pd.read_csv('bikeway_node_gmns.csv')
merge_link_df = pd.read_csv('bikeway_link_gmns.csv')

updated_base_node_df, updated_merge_node_df, updated_base_link_df, updated_merge_link_df = network_update(
    base_node_df, base_link_df, merge_node_df, merge_link_df
)

updated_base_node_df.to_csv('updated_nodes.csv', index=False)
updated_merge_node_df.to_csv('updated_bikeway_nodes.csv', index=False)
updated_base_link_df.to_csv('updated_links.csv', index=False)
updated_merge_link_df.to_csv('updated_bikeway_links.csv', index=False)